In [ ]:
# notebooks/01_analysis.ipynb

import xarray as xr
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# --- 1. Load the "Cube" (Multidimensional Data) ---
# This proves you can handle NetCDF, not just CSVs.
# We use the relative path to find the file we downloaded earlier.
file_path = '../data/raw/era5_nordic_temp.nc'

if not os.path.exists(file_path):
    print(f"Error: File not found at {file_path}")
    print("Please run 'src/ingest_data.py' first.")
else:
    ds = xr.open_dataset(file_path)

    # Show the "0.1%" insight: Explicitly mention dimensions
    print("Data loaded successfully.")
    print(f"Data Dimensions: {ds.dims}") 
    # Output implies: (time, latitude, longitude)

    # --- 2. The "Masking" Challenge ---
    # The ad asks for Python skills. 
    # We need to turn this map into a time-series for specific countries.
    # For this demo, we will use a "Bounding Box" approach to approximate Norway.
    # (Note: In production, we would use a Shapefile, but this proves the concept)

    # Approximate Bounding Box for Southern Norway (Population Center)
    # Lat: 58-62, Lon: 5-12
    norway_subset = ds.sel(latitude=slice(62, 58), longitude=slice(5, 12))

    # --- 3. The Aggregation (Reduce Dimensions) ---
    # We calculate the mean temperature over the area for each month
    # This collapses (Lat, Lon) -> Time
    # 't2m' is the variable name for "2 metre temperature" in ERA5
    norway_temp = norway_subset['t2m'].mean(dim=['latitude', 'longitude'])

    # Convert Kelvin to Celsius (Essential domain knowledge check!)
    norway_temp_c = norway_temp - 273.15

    # --- 4. Visualize (The "Result") ---
    # This graph must be visible in the saved notebook for the recruiter to see
    plt.figure(figsize=(12, 6))
    norway_temp_c.plot()
    plt.title("Reproduced ERA5 Temperature History: Southern Norway (1990-2019)")
    plt.ylabel("Temperature (°C)")
    plt.grid(True, alpha=0.3)
    plt.show()

    # --- 5. Export for Modeling ---
    # Convert this xarray Series to a Pandas DataFrame for the Bayesian model in Notebook 02
    df_weather = norway_temp_c.to_dataframe(name='temperature').reset_index()
    
    # Extract just Year/Month or Year for merging
    # The time column is usually datetime64
    df_weather['year'] = df_weather['time'].dt.year
    
    # We aggregate to Yearly average to match the GBD data structure
    df_yearly = df_weather.groupby('year')['temperature'].mean().reset_index()

    # Create directory if it doesn't exist (Safety first)
    os.makedirs('../data/processed', exist_ok=True)
    
    df_yearly.to_csv('../data/processed/norway_weather_clean.csv', index=False)
    print("Weather data processed and saved to '../data/processed/norway_weather_clean.csv'.")
    print(df_yearly.head())